In [9]:
# Importando as bibliotecas essenciais para análise de dados
import pandas as pd
import numpy as np

# Configura uma semente aleatória para que os dados gerados sejam sempre os mesmos
np.random.seed(42)
print("Bibliotecas importadas com sucesso!")


Bibliotecas importadas com sucesso!


In [10]:
num_rotas = 50

# Geração de colunas logísticas baseadas nas dores da vaga da GM
rotas = [f"BR-{np.random.randint(100, 999)}" for _ in range(num_rotas)]
fluxo = np.random.choice(["Inbound", "Outbound", "Containerization"], size=num_rotas, p=[0.4, 0.4, 0.2])
fornecedores = ["Logística TransBrasil", "Swift Cargo", "Nexus Transportes", "Global Shippers", "Rodoviária Sul"]
fornecedor_atual = np.random.choice(fornecedores, size=num_rotas)

# Custos atuais e simulação de concorrência de preços (RFQ)
custo_atual = np.random.randint(5000, 35000, size=num_rotas)
cotacao_a = (custo_atual * np.random.uniform(0.82, 1.05, size=num_rotas)).astype(int)
cotacao_b = (custo_atual * np.random.uniform(0.80, 1.02, size=num_rotas)).astype(int)

# Variáveis para análise de risco (combustível e vencimento de contratos)
consumo_diesel = (custo_atual * np.random.uniform(0.12, 0.18, size=num_rotas)).astype(int)
dias_vencimento = np.random.randint(-15, 120, size=num_rotas)

# Criando o DataFrame Bruto
df_bruto = pd.DataFrame({
    "ID_Rota": rotas,
    "Tipo_Fluxo": fluxo,
    "Fornecedor_Actual": fornecedor_atual,
    "Custo_Atual_RS": custo_atual,
    "Cotacao_Fornecedor_A_RS": cotacao_a,
    "Cotacao_Fornecedor_B_RS": cotacao_b,
    "Consumo_Diesel_Litros": consumo_diesel,
    "Dias_Para_Expirar_Contrato": dias_vencimento
})

# Salva o arquivo bruto na sua pasta para histórico de governança
df_bruto.to_csv("dados_logisticos_GM_brutos.csv", index=False)
print("Arquivo bruto 'dados_logisticos_GM_brutos.csv' gerado!")
# Mostra as primeiras 5 linhas na tela
df_bruto.head()


Arquivo bruto 'dados_logisticos_GM_brutos.csv' gerado!


,ID_Rota,Tipo_Fluxo,Fornecedor_Actual,Custo_Atual_RS,Cotacao_Fornecedor_A_RS,Cotacao_Fornecedor_B_RS,Consumo_Diesel_Litros,Dias_Para_Expirar_Contrato
0,BR-202,Containerization,Swift Cargo,21157,20314,20304,3082,37
1,BR-535,Outbound,Swift Cargo,15173,14196,15133,2700,44
2,BR-960,Containerization,Logística TransBrasil,26834,22321,26704,4771,92
3,BR-370,Outbound,Global Shippers,23047,20375,22391,3945,-11
4,BR-206,Outbound,Logística TransBrasil,31105,32003,29277,4282,87


In [ ]:
# 1. Carrega o arquivo gerado na célula anterior
df_processado = pd.read_csv("dados_logisticos_GM_brutos.csv")

# 2. Automação do RFQ: Identifica a menor proposta comercial entre os concorrentes
df_processado["Melhor_Cotacao_RS"] = df_processado[["Cotacao_Fornecedor_A_RS", "Cotacao_Fornecedor_B_RS"]].min(axis=1)

# 3. Cálculo do Saving Financeiro Realizado (Diferença entre o custo antigo e o novo)
df_processado["Saving_RS"] = df_processado["Custo_Atual_RS"] - df_processado["Melhor_Cotacao_RS"]

# 4. Análise de Risco de Contratos (Governança e Compliance)
df_processado["Status_Contrato"] = df_processado["Dias_Para_Expirar_Contrato"].apply(
    lambda x: "Expirado" if x < 0 else ("Urgente: Vence em 30d" if x <= 30 else "Regular")
)

# 5. Exporta a base final perfeitamente mastigada para o Power BI
df_processado.to_csv("dados_logisticos_GM_processados.csv", index=False) 
print("Base tratada e exportada com sucesso como 'dados_logisticos_GM_processados.csv'!")
# Mostra as primeiras 5 linhas do resultado final
df_processado.head()


Base tratada e exportada com sucesso como 'dados_logisticos_GM_processados.csv'!


,ID_Rota,Tipo_Fluxo,Fornecedor_Actual,Custo_Atual_RS,Cotacao_Fornecedor_A_RS,Cotacao_Fornecedor_B_RS,Consumo_Diesel_Litros,Dias_Para_Expirar_Contrato,Melhor_Cotacao_RS,Saving_RS,Status_Contrato
0,BR-202,Containerization,Swift Cargo,21157,20314,20304,3082,37,20304,853,Regular
1,BR-535,Outbound,Swift Cargo,15173,14196,15133,2700,44,14196,977,Regular
2,BR-960,Containerization,Logística TransBrasil,26834,22321,26704,4771,92,22321,4513,Regular
3,BR-370,Outbound,Global Shippers,23047,20375,22391,3945,-11,20375,2672,Expirado
4,BR-206,Outbound,Logística TransBrasil,31105,32003,29277,4282,87,29277,1828,Regular
